# 🖼️ CNN-Autoencoder for Brain MRI Anomaly Detection

## 📋 Overview

This notebook implements a **Convolutional Neural Network (CNN) Autoencoder** for anomaly detection in brain MRI scans.

### Model Architecture
- **Type**: Convolutional Autoencoder
- **Input**: 128×128 grayscale MRI slice
- **Encoder**: 4 Conv2D layers with MaxPooling
- **Decoder**: 4 ConvTranspose2D layers for upsampling
- **Latent**: 8×8×256 = 16,384 features

### Key Features
- **Spatial feature extraction** with convolutional layers
- **Translation equivariance** (but NOT rotation equivariant)
- **Encoder-decoder architecture** preserves spatial structure
- **Better than baseline** for capturing local patterns

### Expected Performance
- **AUROC**: ~0.82-0.87 (better than baseline)
- **Training Time**: ~30-40 minutes on Colab GPU

---

## 1️⃣ Setup (Copy from Baseline - Same Setup Code)

In [ ]:
# Mount Drive, install packages, import libraries, setup device, define paths
# (Use exact same cells as Baseline notebook - cells 2-6)

from google.colab import drive
drive.mount('/content/drive')

!pip install pytorch-msssim -q

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from pytorch_msssim import MS_SSIM
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, auc
from glob import glob
import os
import time
from tqdm import tqdm

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Device: {device}")

BASE_PATH = "/content/drive/MyDrive/symAD-ECNN"
IXI_PATH = f"{BASE_PATH}/data/processed_ixi/resized_ixi"
BRATS_PATH = f"{BASE_PATH}/data/brats2021_test"
MODEL_PATH = f"{BASE_PATH}/models/saved_models"
RESULTS_PATH = f"{BASE_PATH}/results"

os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(RESULTS_PATH, exist_ok=True)

print("✅ Setup complete!")

## 2️⃣ Data Loading (Same as Baseline)

In [ ]:
# Dataset, file loading, dataloaders (Same as Baseline)

class MRIDataset(Dataset):
    def __init__(self, file_list):
        self.files = file_list
    def __len__(self):
        return len(self.files)
    def __getitem__(self, idx):
        img = np.load(self.files[idx])
        img = np.expand_dims(img, 0)
        return torch.from_numpy(img).float(), torch.from_numpy(img).float()

ixi_files = sorted(glob(f"{IXI_PATH}/*.npy"))
brats_files = sorted(glob(f"{BRATS_PATH}/*.npy"))
train_files, val_files = train_test_split(ixi_files, test_size=0.1, random_state=42)

BATCH_SIZE = 32
train_loader = DataLoader(MRIDataset(train_files), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(MRIDataset(val_files), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(MRIDataset(brats_files), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"✅ Data loaded: Train={len(train_files)}, Val={len(val_files)}, Test={len(brats_files)}")

## 3️⃣ CNN-Autoencoder Model Architecture

In [ ]:
class CNNAutoencoder(nn.Module):
    """
    Convolutional Autoencoder
    
    Encoder: 128×128 -> 64×64 -> 32×32 -> 16×16 -> 8×8
    Decoder: 8×8 -> 16×16 -> 32×32 -> 64×64 -> 128×128
    """
    
    def __init__(self, latent_dim=256):
        super(CNNAutoencoder, self).__init__()
        
        # Encoder: Progressive downsampling
        self.encoder = nn.Sequential(
            # 128×128×1 -> 128×128×32
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(True),
            nn.MaxPool2d(2, 2),  # 64×64
            
            # 64×64×32 -> 64×64×64
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            nn.MaxPool2d(2, 2),  # 32×32
            
            # 32×32×64 -> 32×32×128
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.MaxPool2d(2, 2),  # 16×16
            
            # 16×16×128 -> 16×16×256
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.MaxPool2d(2, 2)  # 8×8
        )
        
        # Latent space
        self.flatten = nn.Flatten()
        self.fc_encode = nn.Linear(8 * 8 * 256, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, 8 * 8 * 256)
        
        # Decoder: Progressive upsampling
        self.decoder = nn.Sequential(
            # 8×8×256 -> 16×16×256
            nn.ConvTranspose2d(256, 256, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            
            # 16×16×256 -> 32×32×128
            nn.ConvTranspose2d(256, 128, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            
            # 32×32×128 -> 64×64×64
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            
            # 64×64×64 -> 128×128×32
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(True),
            
            # 128×128×32 -> 128×128×1
            nn.Conv2d(32, 1, kernel_size=3, stride=1, padding=1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        # Encode
        features = self.encoder(x)
        
        # Latent
        batch_size = features.size(0)
        flat = self.flatten(features)
        z = self.fc_encode(flat)
        
        # Decode
        decoded_flat = self.fc_decode(z)
        decoded_features = decoded_flat.view(batch_size, 256, 8, 8)
        x_recon = self.decoder(decoded_features)
        
        return x_recon
    
    def get_latent(self, x):
        features = self.encoder(x)
        flat = self.flatten(features)
        return self.fc_encode(flat)

model = CNNAutoencoder().to(device)
total_params = sum(p.numel() for p in model.parameters())

print("🖼️ CNN-Autoencoder Created!")
print(f"   Total parameters: {total_params:,}")
print(f"   Model size: ~{total_params * 4 / 1024**2:.2f} MB")

## 4️⃣ Training (Same loss function, optimizer, training loop as Baseline)

In [ ]:
# Copy training cells from Baseline (cells 15-20)
# Loss function, optimizer, train_epoch, validate, training loop, plot curves

class CombinedLoss(nn.Module):
    def __init__(self, alpha=0.84):
        super().__init__()
        self.alpha = alpha
        self.mse = nn.MSELoss()
        self.ms_ssim = MS_SSIM(data_range=1.0, size_average=True, channel=1)
    def forward(self, output, target):
        return self.alpha * self.mse(output, target) + (1 - self.alpha) * (1 - self.ms_ssim(output, target))

criterion = CombinedLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True, min_lr=1e-6)

def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for data, target in tqdm(dataloader, desc='Training'):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    return running_loss / len(dataloader)

def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for data, target in tqdm(dataloader, desc='Validation'):
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)
            running_loss += loss.item()
    return running_loss / len(dataloader)

print("✅ Training setup complete!")

In [ ]:
# Main training loop
NUM_EPOCHS = 100
train_losses, val_losses = [], []
best_val_loss = float('inf')

print("🚀 Starting CNN-Autoencoder Training...")

for epoch in range(NUM_EPOCHS):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, val_loader, criterion, device)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    scheduler.step(val_loss)
    
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] Train: {train_loss:.6f}, Val: {val_loss:.6f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), f'{MODEL_PATH}/cnn_best.pth')
        print(f"   ✅ Best model saved!")

print("🎉 Training complete!")

# Plot curves
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train')
plt.plot(val_losses, label='Val')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('CNN-Autoencoder Training')
plt.savefig(f'{RESULTS_PATH}/cnn_training.png')
plt.show()

## 5️⃣ Evaluation and Results

In [ ]:
# Evaluation (Same as Baseline - calculate errors, AUROC, visualizations)

def calculate_reconstruction_error(model, dataloader, device):
    model.eval()
    errors = []
    with torch.no_grad():
        for data, _ in tqdm(dataloader):
            data = data.to(device)
            recon = model(data)
            mse = nn.functional.mse_loss(recon, data, reduction='none')
            mse = mse.view(mse.size(0), -1).mean(dim=1)
            errors.extend(mse.cpu().numpy())
    return np.array(errors)

normal_errors = calculate_reconstruction_error(model, val_loader, device)
anomaly_errors = calculate_reconstruction_error(model, test_loader, device)

y_true = np.concatenate([np.zeros(len(normal_errors)), np.ones(len(anomaly_errors))])
y_scores = np.concatenate([normal_errors, anomaly_errors])

auroc = roc_auc_score(y_true, y_scores)
precision, recall, _ = precision_recall_curve(y_true, y_scores)
auprc = auc(recall, precision)

print(f"📈 CNN-Autoencoder Performance:")
print(f"   AUROC: {auroc:.4f}")
print(f"   AUPRC: {auprc:.4f}")

# Plot
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.hist(normal_errors, bins=50, alpha=0.7, label='Normal', density=True)
plt.hist(anomaly_errors, bins=50, alpha=0.7, label='Anomaly', density=True)
plt.xlabel('Reconstruction Error')
plt.ylabel('Density')
plt.legend()
plt.title('Error Distribution')

plt.subplot(1, 2, 2)
fpr, tpr, _ = roc_curve(y_true, y_scores)
plt.plot(fpr, tpr, label=f'CNN-AE (AUROC={auroc:.4f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.legend()
plt.title('ROC Curve')

plt.tight_layout()
plt.savefig(f'{RESULTS_PATH}/cnn_evaluation.png')
plt.show()

# Save results
import json
results = {'model': 'CNN-Autoencoder', 'auroc': float(auroc), 'auprc': float(auprc), 'best_val_loss': float(best_val_loss)}
with open(f'{RESULTS_PATH}/cnn_results.json', 'w') as f:
    json.dump(results, f, indent=4)

print("✅ CNN-Autoencoder evaluation complete!")